# 01 · Ingesta y adquisición de datos
**Proyecto:** Minería de datos · Contratación pública de la Alcaldía de Pereira

**Pregunta de negocio (Ficha):** ¿Qué procesos de contratación de la Alcaldía de Pereira registrados en SECOP II entre 2022 y 2025 presentan señales de concentración en proveedores o montos atípicos, para que la Oficina de Control Interno priorice cuáles revisar primero?

**Objetivo de este notebook:** leer las fuentes, verificar el cruce y documentar el dataset resultante.

## Fuentes
| Archivo | Origen | Periodo | Contenido |
|---|---|---|---|
| Vista-_Contratos_Alcaldia_de_Pereira_2020-2023.xlsx | SECOP II · Contratos Electrónicos (datos.gov.co, `jbjy-vk9h`) | 2020-2023 | Contratos de la Alcaldía de Pereira |
| Vista-_Contratos_Alcaldia_de_Pereira_2024-2026.xlsx | SECOP II · Contratos Electrónicos (datos.gov.co, `jbjy-vk9h`) | 2024-2026 | Contratos de la Alcaldía de Pereira |

**Nota metodológica:** ambos archivos provienen de una misma exportación (2020-2026) que se dividió en dos periodos para comparar las dos administraciones. El cruce con el dataset SECOP II · Procesos de Contratación (`p6dx-8zbt`) queda como siguiente paso.

In [ ]:
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. Lectura de las fuentes
Se leen ambos archivos con `pd.read_excel`. La función `buscar` los localiza en la carpeta actual (Colab) o en `data/raw` (repositorio). Se verifica con `.shape` (filas, columnas) y `.head()`.

In [ ]:
def buscar(nombre):
    for base in [".", "data/raw", "../data/raw"]:
        ruta = Path(base) / nombre
        if ruta.exists():
            return ruta
    raise FileNotFoundError(f"No se encontró {nombre}. Súbelo a Colab o colócalo en data/raw.")

ruta1 = buscar("Vista-_Contratos_Alcaldia_de_Pereira_2020-2023.xlsx")
ruta2 = buscar("Vista-_Contratos_Alcaldia_de_Pereira_2024-2026.xlsx")

df1 = pd.read_excel(ruta1)
df2 = pd.read_excel(ruta2)

print("Fuente 1 (2020-2023):", df1.shape)
print("Fuente 2 (2024-2026):", df2.shape)

In [ ]:
display(df1.head())
display(df2.head())

In [ ]:
print("Columnas fuente 1:\n", list(df1.columns), "\n")
print("Columnas fuente 2:\n", list(df2.columns), "\n")

comunes = set(df1.columns) & set(df2.columns)
print("Columnas en común:", len(comunes))
print("Solo en fuente 1:", set(df1.columns) - set(df2.columns))
print("Solo en fuente 2:", set(df2.columns) - set(df1.columns))

df1.info()

## 2. Normalización de columnas
Se pasan los nombres a minúsculas, sin tildes ni espacios, y se agrega `fuente_periodo` para saber de qué archivo viene cada fila.

In [ ]:
def normalizar(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(" ", "_")
                  .str.replace("[áàä]", "a", regex=True)
                  .str.replace("[éèë]", "e", regex=True)
                  .str.replace("[íìï]", "i", regex=True)
                  .str.replace("[óòö]", "o", regex=True)
                  .str.replace("[úùü]", "u", regex=True))
    return df

df1 = normalizar(df1); df1["fuente_periodo"] = "2020-2023"
df2 = normalizar(df2); df2["fuente_periodo"] = "2024-2026"
print(list(df1.columns))

## 3. Cruce de las fuentes
**Columna en común (llave):** `id_contrato`, identificador único de cada contrato.

Primero se revisa la calidad de la llave (nulos y duplicados). Luego se hace un `merge` (outer) para verificar cuántos contratos aparecen en un solo periodo o en ambos, y por último se construye el dataset final. Se verifica el número de filas **antes y después** de cada paso.

In [ ]:
LLAVE = "id_contrato"

print("Nulos llave  -> df1:", df1[LLAVE].isna().sum(), "| df2:", df2[LLAVE].isna().sum())
print("Duplicados   -> df1:", df1[LLAVE].duplicated().sum(), "| df2:", df2[LLAVE].duplicated().sum())
print("Contratos presentes en AMBOS periodos:", len(set(df1[LLAVE]) & set(df2[LLAVE])))

In [ ]:
print("Filas ANTES -> df1:", len(df1), "| df2:", len(df2), "| suma:", len(df1) + len(df2))

cruce = pd.merge(df1[[LLAVE, "fuente_periodo"]],
                 df2[[LLAVE, "fuente_periodo"]],
                 on=LLAVE, how="outer",
                 suffixes=("_a", "_b"), indicator=True)

print("Filas DESPUÉS del merge:", len(cruce))
print(cruce["_merge"].value_counts())
# left_only = solo en 2020-2023 | right_only = solo en 2024-2026 | both = en los dos

In [ ]:
df_final = pd.concat([df1, df2], ignore_index=True)
print("Filas tras concat:", len(df_final))

df_final = df_final.drop_duplicates(subset=LLAVE, keep="last")
print("Filas finales (sin ID repetido):", len(df_final))
assert df_final[LLAVE].is_unique

**Interpretación del cruce:** Los dos archivos tienen el mismo esquema (14 columnas idénticas) y son partes de una misma exportación de SECOP II · Contratos Electrónicos dividida por periodo. La llave `id_contrato` no tiene nulos ni duplicados en ninguna de las dos, por lo que es válida.

El `merge` outer por `id_contrato` dio 22.522 contratos `left_only`, 17.753 `right_only` y 0 `both`. Como el corte por periodo era una partición, el resultado esperado era cero solapamiento, y el `merge` sirve como verificación de integridad: no se perdieron ni se duplicaron contratos. El dataset final se construyó con `pd.concat` (22.522 + 17.753 = 40.275 filas) y `drop_duplicates` sobre la llave no eliminó ninguna fila.

El dataset resultante reúne los contratos de la Alcaldía de Pereira entre 2020 y 2026 y contiene las variables necesarias para la pregunta de la Ficha (`proveedor_adjudicado`, `documento_proveedor`, `valor_del_contrato`, `modalidad_de_contratacion`). El cruce con el dataset de Procesos (`p6dx-8zbt`) por `proceso_de_compra` queda como siguiente paso.

## 4. Tipos de dato
Se convierten fechas y valores a su tipo correcto. `documento_proveedor` se deja como texto porque es un identificador, no una cantidad.

In [ ]:
df_final["fecha_de_inicio_del_contrato"] = pd.to_datetime(df_final["fecha_de_inicio_del_contrato"], errors="coerce")
df_final["fecha_de_fin_del_contrato"]    = pd.to_datetime(df_final["fecha_de_fin_del_contrato"], errors="coerce")
df_final["valor_del_contrato"]  = pd.to_numeric(df_final["valor_del_contrato"], errors="coerce")
df_final["documento_proveedor"] = df_final["documento_proveedor"].astype(str)

print(df_final.shape)
df_final.dtypes

In [ ]:
# Para describir bien las columnas categóricas en el diccionario
print(df_final["estado_contrato"].unique())
print(df_final["tipo_de_contrato"].unique())
print(df_final["modalidad_de_contratacion"].unique())

## 5. Diccionario de datos
Descripción de las columnas del dataset cruzado: tipo de dato, significado y fuente.

In [ ]:
FUENTE = "SECOP II · Contratos Electrónicos (datos.gov.co, jbjy-vk9h)"

significados = {
    "proceso_de_compra": ("Código del proceso de contratación del que se origina el contrato (llave hacia el dataset de Procesos)", FUENTE),
    "id_contrato": ("Identificador único del contrato (llave del cruce)", FUENTE),
    "referencia_del_contrato": ("Número o referencia interna asignada al contrato", FUENTE),
    "estado_contrato": ("Estado del contrato en la plataforma (Cerrado, En ejecución, Modificado, terminado, cedido, Cancelado, Aprobado, Borrador, Suspendido, enviado Proveedor)", FUENTE),
    "descripcion_del_proceso": ("Descripción general del proceso de compra", FUENTE),
    "tipo_de_contrato": ("Categoría del contrato (Prestación de servicios, Obra, Suministros, Consultoría, Arrendamiento, Interventoría, Seguros, Concesión, entre otras)", FUENTE),
    "modalidad_de_contratacion": ("Modalidad de selección (Contratación directa, Mínima cuantía, Licitación pública, Selección abreviada, Concurso de méritos, Régimen especial)", FUENTE),
    "fecha_de_inicio_del_contrato": ("Fecha de inicio de la ejecución del contrato", FUENTE),
    "fecha_de_fin_del_contrato": ("Fecha de terminación prevista del contrato", FUENTE),
    "documento_proveedor": ("NIT o cédula del proveedor adjudicado", FUENTE),
    "proveedor_adjudicado": ("Nombre del proveedor o contratista", FUENTE),
    "valor_del_contrato": ("Valor del contrato en pesos colombianos (COP)", FUENTE),
    "urlproceso": ("Enlace al proceso en la plataforma de contratación", FUENTE),
    "objeto_del_contrato": ("Descripción de lo que se contrata", FUENTE),
    "fuente_periodo": ("Archivo/periodo de origen del registro (columna creada en el notebook)", "Creada en este notebook"),
}

diccionario = pd.DataFrame({
    "columna": df_final.columns,
    "tipo_dato": df_final.dtypes.astype(str).values,
    "significado": [significados.get(c, ("", ""))[0] for c in df_final.columns],
    "fuente": [significados.get(c, ("", ""))[1] for c in df_final.columns],
})
diccionario

## 6. Verificaciones de calidad
Se comprueba que el rango de fechas de cada archivo corresponda a su periodo, se buscan valores nulos y se revisan fechas inconsistentes (inicio posterior a hoy o fin anterior al inicio).

In [ ]:
print(df_final.groupby("fuente_periodo")["fecha_de_inicio_del_contrato"].agg(["min", "max", "count"]))
print("\nNulos por columna:")
print(df_final.isna().sum()[lambda x: x > 0])

In [ ]:
hoy = pd.Timestamp.today()
futuros = df_final["fecha_de_inicio_del_contrato"] > hoy
print("Contratos con inicio posterior a hoy:", futuros.sum())
print(df_final.loc[futuros, "estado_contrato"].value_counts())
print("\nContratos con fin anterior al inicio:",
      (df_final["fecha_de_fin_del_contrato"] < df_final["fecha_de_inicio_del_contrato"]).sum())

## 7. Subconjunto de análisis 2022-2025
La pregunta de la Ficha cubre 2022-2025, así que se crea un subconjunto por fecha de inicio. El dataset completo (2020-2026) se conserva para contexto.

In [ ]:
inicio = df_final["fecha_de_inicio_del_contrato"]
df_analisis = df_final[(inicio >= "2022-01-01") & (inicio <= "2025-12-31")].copy()

print("Dataset completo:", df_final.shape)
print("Subconjunto 2022-2025:", df_analisis.shape)
print(df_analisis["fuente_periodo"].value_counts())

## 8. Exportación
Se guardan el dataset cruzado, el subconjunto de análisis y el diccionario de datos para las siguientes etapas (EDA y detección de anomalías).

In [ ]:
df_final.to_csv("contratos_pereira_2020_2026.csv", index=False)
df_analisis.to_csv("contratos_pereira_2022_2025.csv", index=False)
diccionario.to_csv("diccionario_datos.csv", index=False)
print("Archivos guardados")

## 9. Conclusiones de la etapa de adquisición
- Se leyeron 2 archivos de SECOP II · Contratos Electrónicos con 22.522 (2020-2023) y 17.753 (2024-2026) filas, ambos con 14 columnas idénticas.
- El `merge` por `id_contrato` confirmó que no hay contratos compartidos entre periodos (`both = 0`) y que el número de filas se conserva: 40.275 antes y después.
- El dataset final tiene 40.275 filas y 15 columnas, sin valores nulos. El subconjunto 2022-2025, que corresponde al alcance de la Ficha, tiene 24.779 filas.
- Hallazgos para la limpieza: `estado_contrato` mezcla mayúsculas y minúsculas (`Cerrado`, `terminado`, `cedido`); `modalidad_de_contratacion` tiene categorías casi duplicadas (p. ej. `Contratación directa` y `Contratación Directa (con ofertas)`); `tipo_de_contrato` incluye categorías ambiguas como `Otro` y `Decreto 092 de 2017`; hay estados como `Borrador` y `Aprobado` que pueden no ser contratos formalizados; 2 contratos tienen fecha de inicio posterior a la fecha de consulta y 3 tienen fecha de fin anterior al inicio.
- Próximos pasos: cruzar con SECOP II · Procesos de Contratación (`p6dx-8zbt`) por `proceso_de_compra`, limpiar categorías y avanzar al análisis exploratorio (`02_eda`).